In [16]:
from rich import print as pprint

# Metaclasses

Metaclasse é a “classe das classes”.
Em Python, toda classe é um objeto criado por uma metaclasse. 

A metaclasse padrão é `type`. Quando você escreve:
```python
class Pessoa:
    pass
```

o Python chama a metaclasse (`type`, por padrão) para construir o objeto-classe `Pessoa`.

O que a metaclasse controla?

Ela define **como as classes são criadas e inicializadas**. O ciclo é:

1. `__prepare__(name, bases, **kwargs)` → cria o namespace (dict) onde o corpo da classe será executado.

2. Executa o corpo da classe dentro desse namespace.

3. `__new__(mcls, name, bases, namespace)` → cria o objeto-classe.

4. `__init__(cls, name, bases, namespace)` → inicializa o objeto-classe.

5. Depois, na hora de instanciar (`Pessoa()`), é a metaclasse que fornece `__call__`, que por sua vez chama `cls.__new__` e `cls.__init__`.

Resumindo: 
- **classe** controla instâncias;
- **metaclasse** controla classes.

## Como classes sao criadas


Uma classe é uma instância de `type`. E `type` é uma classe em si, então ela é chamável (com alguns argumentos) e é usada para criar classes, instâncias da classe `type`. Por isso o tipo de uma classe é `type`


partes de uma classe:

![Partes classe](imgs/partes_classe.jpg)


1. o corpo da classe e extraido
2. um novo dicionario e criado - ele vai ser o *namespace* da nova classe
3. o codigo do corpo da classe é executado dentro de *namespace*, assim preenchendo-o
4. dentro do *namespace* haverao os simbolos:
   - age, cujo valor sera 12;
   - name, cujo valor e um objeto *property*;
   - `__init__`, que sera uma funcao
5. uma nova instancia `type`é criada usando o nome da classe, a(s) classe(s) base(s) (no caso *object*) e o dicionario preenchido

Olhemos inicialmente para o método built-in `exec`:

Vamos experimentar com um exemplo simples primeiro:

### `exec` e compilacao em python

In [22]:
pprint(help(exec))

Help on built-in function exec in module builtins:

exec(source, globals=None, locals=None, /)
    Execute the given source in the context of globals and locals.
    
    The source may be a string representing one or more Python statements
    or a code object as returned by compile().
    The globals must be a dictionary and locals can be any mapping,
    defaulting to the current globals and locals.
    If only globals is given, locals defaults to it.



None

In [11]:
namespace = {}

exec('''
a = 10
b = 20
''', globals(), namespace)

In [12]:
namespace

{'a': 10, 'b': 20}

In [13]:
exec('''
def add(a, b):
    return a + b
    
def mul(a, b):
    return a * b
''', globals(), namespace)

In [14]:
namespace

{'a': 10,
 'b': 20,
 'add': <function __main__.add(a, b)>,
 'mul': <function __main__.mul(a, b)>}

In [15]:
namespace['add'](10, 20)

30

As funções definidas nesse escopo não sabem realmente nada sobre o que mais está naquele escopo - essas funções são criadas independentemente do dicionário no qual são inseridas.

Então, é assim que vamos "executar" o corpo da classe no contexto do dicionário de namespace da classe.

Também precisaremos criar uma nova instância type, então vamos ver qual é a assinatura do construtor type:

### Criando uma classe manualmente. 

In [21]:
class_name = 'Person'

class_body = """
age = 12
name = property(fget=lambda self: self._name)

def __init__(self, name):
    self._name = name
"""
class_bases = ()  # defaults to object

class_dict = {}

exec(class_body, globals(), class_dict)

Person = type(class_name, class_bases, class_dict)

pprint(f"->Person: {Person}")
pprint(f"->type(Person): {type(Person)}")
pprint(f"->vars(Person): {vars(Person)}")
pprint(f"->class_dict: {class_dict}")

->Person: <class '__main__.Person'>

->type(Person): <class 'type'>

->vars(Person): {'age': 12, 'name': <property object at 0x0000018558EAEC50>, '__init__': <function __init__ at 
0x0000018558EAA680>, '__module__': '__main__', '__dict__': <attribute '__dict__' of 'Person' objects>, 
'__weakref__': <attribute '__weakref__' of 'Person' objects>, '__doc__': None}

->class_dict: {'age': 12, 'name': <property object at 0x0000018558EAEC50>, '__init__': <function __init__ at 
0x0000018558EAA680>}

Passo a passo do codigo de criacao de classe:

1. **class_bases = ()**
   
   * Tupla vazia → Person herda de object (a classe base padrão do Python).
   
   * Equivalente a escrever `class Person: ...` sem herança explícita.

2. **exec(class_body, ...)**
   
   * Executa a string `class_body` como código Python.
   
   * Preenche `class_dict` com atributos/métodos (`age`, `name`, `init`).

3. **type(...)**
   
   * Cria dinamicamente uma classe chamada `Person`.
   
   * Parâmetros:
        - `class_name`: Nome da classe ("Person").
        - `class_bases`: Tupla de classes base (vazia → usa `object`).
        - `class_dict`: Atributos/métodos do `exec()`.

In [5]:
class_name = 'Circle'

class_body = """
def __init__(self, x, y, r):
    self.x = x
    self.y = y
    self.r = r

def area(self):
    return math.pi * self.r ** 2
"""
class_bases = ()  # defaults to object

class_dict = {}

exec(class_body, globals(), class_dict)

Circle = type(class_name, class_bases, class_dict)

In [6]:
class_dict

{'__init__': <function __main__.__init__(self, x, y, r)>,
 'area': <function __main__.area(self)>}

In [10]:
Circle, type(Circle), vars(Circle)

(__main__.Circle,
 type,
 mappingproxy({'__init__': <function __main__.__init__(self, x, y, r)>,
               'area': <function __main__.area(self)>,
               '__module__': '__main__',
               '__dict__': <attribute '__dict__' of 'Circle' objects>,
               '__weakref__': <attribute '__weakref__' of 'Circle' objects>,
               '__doc__': None}))

## `type` vs `object`

Importante salientar a diferença entre `type` e `object`.


| ASPECTO       | `object`                                    | `type`                             |
|---------------|------------------------------------------|-----------------------------------------|
| Papel         | Classe base para todos os objetos/classes. | Metaclasse que cria classes.            |
| Herança       | Raiz da hierarquia de classes.           | Herda de `object` (como todas as classes). |
| Uso           | Usado em definições de classe (ex: `class MinhaClasse(object):`). | Usado para gerar classes dinamicamente (ex: `MinhaClasse = type("MinhaClasse", (), {})`). |
| Identidade    | Uma instância de `type` (já que é uma classe). | Uma instância de si mesma (`type(type)` é `type`). |

![Visual Relationship](imgs/esquema_type_object.png)

In [25]:
help(object)

Help on class object in module builtins:

class object
 |  The base class of the class hierarchy.
 |  
 |  When called, it accepts no arguments and returns a new featureless
 |  instance that has no instance attributes and cannot be given any.
 |  
 |  Built-in subclasses:
 |      anext_awaitable
 |      async_generator
 |      async_generator_asend
 |      async_generator_athrow
 |      ... and 88 other subclasses
 |  
 |  Methods defined here:
 |  
 |  __delattr__(self, name, /)
 |      Implement delattr(self, name).
 |  
 |  __dir__(self, /)
 |      Default dir() implementation.
 |  
 |  __eq__(self, value, /)
 |      Return self==value.
 |  
 |  __format__(self, format_spec, /)
 |      Default object formatter.
 |  
 |  __ge__(self, value, /)
 |      Return self>=value.
 |  
 |  __getattribute__(self, name, /)
 |      Return getattr(self, name).
 |  
 |  __gt__(self, value, /)
 |      Return self>value.
 |  
 |  __hash__(self, /)
 |      Return hash(self).
 |  
 |  __init__(self, /

In [24]:
help(type)

Help on class type in module builtins:

class type(object)
 |  type(object) -> the object's type
 |  type(name, bases, dict, **kwds) -> a new type
 |  
 |  Methods defined here:
 |  
 |  __call__(self, /, *args, **kwargs)
 |      Call self as a function.
 |  
 |  __delattr__(self, name, /)
 |      Implement delattr(self, name).
 |  
 |  __dir__(self, /)
 |      Specialized __dir__ implementation for types.
 |  
 |  __getattribute__(self, name, /)
 |      Return getattr(self, name).
 |  
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __instancecheck__(self, instance, /)
 |      Check if an object is an instance.
 |  
 |  __or__(self, value, /)
 |      Return self|value.
 |  
 |  __repr__(self, /)
 |      Return repr(self).
 |  
 |  __ror__(self, value, /)
 |      Return value|self.
 |  
 |  __setattr__(self, name, value, /)
 |      Implement setattr(self, name, value).
 |  
 |  __sizeof__(self, /)
 |      Return mem

In [29]:
print(f"issubclass(Person, object): {issubclass(Person, object)}")  # True (all classes inherit from object)
print(f"isinstance(Person, type): {isinstance(Person, type)}")    # True (Person is an instance of type)
pprint(f"Person.__bases__: {Person.__bases__}")

issubclass(Person, object): True
isinstance(Person, type): True


Person.__bases__: (<class 'object'>,)

### Resumo

- `object`: A classe base fundamental para tudo em Python.
- `type`: A metaclasse que constrói classes (incluindo a própria `object`).

- O código de `Person` e `Circle` imitam o processo interno de criação de classes do Python usando `type` como fábrica, com `object` como base implícita.

## Heranca de `type`

Anteriormente, vimos como as classes podem ser criadas chamando a classe `type`.

Mas e se quisermos usar algo diferente de `type` para construir classes?

Como `type` é uma classe, talvez possamos definir uma classe que herde de `type` (para que possamos aproveitar o processo real de criação de `type`) e substituir algumas coisas que nos permitiriam injetar algo no processo de criação da classe.

Aqui queremos interceptar a criação da instância `type` antes que ela seja criada, então gostaríamos de usar o método `__new__`.

Lembre-se de que o método `__new__` basicamente precisa construir e retornar a nova instância. Então faremos as customizações que quisermos, mas, no final, delegaremos para a classe `type` fazer o trabalho real, apenas adicionando os ajustes que quisermos (antes e/ou depois da criação da classe).

Apenas um lembrete rápido de como o método estático `__new__` funciona em geral:

In [1]:
class Test:
    def __new__(cls, *args, **kwargs):
        print(f'New instance of {cls} being created with these values:', args, kwargs)

In [2]:
t = Test(10, 20, kw='a')

New instance of <class '__main__.Test'> being created with these values: (10, 20) {'kw': 'a'}


Bom, é claro que precisamos devolver um objeto a partir da função  `__new__`:

In [3]:
type(t)

NoneType

Vamos criar uma classe geradora customizada por *subclassing* `type`. 

Iremos herdar de `type` e sobrescrever/fazer *override* da função  `__new__` para criar a instancia de classe

In [31]:
import math

class CustomType(type):
    def __new__(cls, name, bases, class_dict):
        # above is the signature that type.__new__ has - 
        # and args are collected and passed by Python when we call a class (to create an instance of that class)
        # we'll see where those actually come from later
        print('Customized type creation!')
        cls_obj = super().__new__(cls, name, bases, class_dict)  # delegate to super (type in this case)
        cls_obj.circ = lambda self: 2 * math.pi * self.r  # basically attaching a function to the class
        return cls_obj

Vamos agora repetir o processo manual de criacao de classe mas usando a classe customizada `CustomType` no lugar da default `type`

In [32]:
class_name = 'Circle'

class_body = """
def __init__(self, x, y, r):
    self.x = x
    self.y = y
    self.r = r

def area(self):
    return math.pi * self.r ** 2
"""
class_bases = ()

class_dict = {}

exec(class_body, globals(), class_dict)

# e a criacao, enfim, da classe Circle:
Circle = CustomType(class_name, class_bases, class_dict)

Customized type creation!


In [33]:
type(Circle)

__main__.CustomType

`Circle` ainda e uma instancia de `type` ja que `CustomType` é uma subclasse de `type`

In [34]:
isinstance(Circle, CustomType), issubclass(CustomType, type)

(True, True)

`Circle` tambem possui os metodos `__init__`e `area`

In [35]:
hasattr(Circle, '__init__'), hasattr(Circle, 'area')

(True, True)

In [36]:
c = Circle(0, 0, 1)
c.area()

3.141592653589793

In [37]:
# e, claro, nos injetamos uma nova funcao circ na classe enquanto nos o construimos no metodo __new__ de CustomType
c.circ()

6.283185307179586

## Exemplo

### Criação padrao de classes/subclasses

- `Person` e `Student` são criados pela metaclasse padrão `type`.
- Tudo o que você personaliza é sobre instâncias (ex: `__init__`, `@property`).
- A criação da classe é básica; nenhum hook é executado além de `type.__new__`/ `type.__init__`.

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

class Student(Person):
    def __init__(self, name, major):
        super().__init__(name)
        self._major = major

    @property
    def major(self):
        return self._major


### Personalizando criação usando metaclasse 

- `Person` é uma instância de `MyType`, não do `type` padrão.
- Como `Student` herda de `Person`, `Student` também usa `MyType` (a metaclasse se `propaga`).
- `MyType.__new__` é executado quando as declarações de classe são processadas (uma vez por classe), imprimindo informações e anexando um atributo de classe `created_by`.
- O comportamento da instância (`__init__`, `@property`) permanece inalterado porque você não sobrescreveu `MyType.__call__`.

In [ ]:
class MyType(type):
    def __new__(mcls, name, bases, cls_dict):
        print(f"Creating class: {name}")
        print(f"Base classes: {bases}")
        print(f"Class dictionary keys: {list(cls_dict.keys())}")
        new_class = super().__new__(mcls, name, bases, cls_dict)
        new_class.created_by = "MyType metaclass"
        return new_class

class Person(metaclass=MyType):
    def __init__(self, name):
        self.name = name

class Student(Person):
    def __init__(self, name, major):
        super().__init__(name)
        self._major = major

    @property
    def major(self):
        return self._major


# `__init_subclass__`

## O que é

- É um hook de classe chamado automaticamente toda vez que uma subclasse é criada.
- A assinatura padrão é `def __init_subclass__(cls, **kwargs)`, e ela vive na(s) classe(s) base.
- Serve para configurar a nova subclasse (`cls`), validar contratos, registrar subclasses, injetar atributos/métodos, etc.
- Ele roda depois que a classe já foi criada, então você não altera o dicionário antes da criação (para isso, metaclass). Mas você pode mutar a classe criada (adicionar atributos, decorar métodos, etc.).
- Em herança múltipla, se mais de um base define `__init_subclass__`, eles são chamados em cadeia via MRO — por isso sempre chame `super().__init_subclass__(**kwargs)` no final (ou começo) do seu override.

Diferença resumida pra metaclasse: 
- metaclasse (`type.__new__`/ `type.__init__`) intercepta a criação da classe (antes de existir).
- `__init_subclass__` é um pós-criação prático, com menos atrito.

## Casos de uso

### Registro automático de subclasses (plugins)

In [13]:
# Perfeito para construir “descoberta” de plugins sem ficar importando manualmente.
class PluginBase:
    registry = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        # chave: nome simples; valor: a própria classe
        PluginBase.registry[cls.__name__] = cls

    def run(self, *args, **kwargs):
        raise NotImplementedError


class CsvExporter(PluginBase):
    def run(self, data):
        return "csv," + ",".join(map(str, data))


class JsonExporter(PluginBase):
    def run(self, data):
        import json
        return json.dumps({"data": data})


# Demonstração
print(PluginBase.registry)    # {'CsvExporter': <class ...>, 'JsonExporter': <class ...>}
print(PluginBase.registry["CsvExporter"]().run([1, 2, 3]))


{'CsvExporter': <class '__main__.CsvExporter'>, 'JsonExporter': <class '__main__.JsonExporter'>}
csv,1,2,3


### Validação de contrato (exigir método com certa assinatura)

In [14]:
# Às vezes você não quer trazer abc.ABC, só garantir que o método exista com a assinatura correta.
import inspect

class HandlerBase:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

        # Exigir método "process(self, data: dict) -> int"
        fn = cls.__dict__.get("process")
        if fn is None:
            raise TypeError(f"{cls.__name__} deve definir um método process(self, data: dict) -> int")

        sig = inspect.signature(fn)
        params = list(sig.parameters.values())
        if not (len(params) == 2 and params[1].annotation is dict):
            raise TypeError("process deve receber (self, data: dict)")
        if sig.return_annotation is inspect._empty or sig.return_annotation is not int:
            raise TypeError("process deve anotar retorno como int")


class MyHandler(HandlerBase):
    def process(self, data: dict) -> int:
        return len(data)


Se quiser combinar com `abc.ABC`/`@abstractmethod`, funciona bem — `__init_subclass__` pode checar apenas quando a classe não é abstrata (ex.: ignorar se `inspect.isabstract(cls)` for `True`).

### Configuração via parâmetros na declaração da classe

Dá para “passar kwargs” ao declarar a subclasse: `class X(Base, strict=True, role="service")`.

In [15]:
class Base:
    def __init_subclass__(cls, *, strict=False, role=None, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.strict = strict
        cls.role = role
        if role not in {None, "service", "worker"}:
            raise ValueError(f"role inválido: {role!r}")


class Service(Base, strict=True, role="service"):
    pass


class Worker(Base, role="worker"):
    pass


print(Service.strict, Service.role)  # True service
print(Worker.strict, Worker.role)    # False worker


True service
False worker


Dica: use `*,` para forçar kwargs nomeados e capturar extras com `**kwargs` (e repassar ao `super()` para cooperar com outros mixins).

### Mixins cooperativos (herança múltipla)

Cada mixin pode configurar algo e todos cooperam chamando `super()`.

In [16]:
class TimestampedMixin:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        import datetime as _dt
        cls.created_at = _dt.datetime.utcnow()


class LabeledMixin:
    def __init_subclass__(cls, label=None, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.label = label or cls.__name__


class Entity(TimestampedMixin, LabeledMixin):
    def __init_subclass__(cls, **kwargs):
        # Pode fazer algo extra e ainda cooperar
        super().__init_subclass__(**kwargs)
        cls.is_entity = True


class User(Entity, label="user-class"):
    pass


print(User.is_entity, User.label, hasattr(User, "created_at"))


True user-class True


### Decorar/envolver métodos automaticamente

Ex.: logar tempo de execução de métodos que começam com `do_`.

In [17]:
import time
import types
from functools import wraps

def timed(f):
    @wraps(f)
    def wrapper(*a, **k):
        t0 = time.perf_counter()
        try:
            return f(*a, **k)
        finally:
            dt = (time.perf_counter() - t0) * 1000
            print(f"[timed] {f.__qualname__} levou {dt:.2f} ms")
    return wrapper


class AutoTimed:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        for name, obj in list(cls.__dict__.items()):
            if name.startswith("do_") and isinstance(obj, (types.FunctionType, classmethod, staticmethod)):
                # compatível com function/classmethod/staticmethod
                if isinstance(obj, (classmethod, staticmethod)):
                    func = obj.__func__
                    wrapped = type(obj)(timed(func))
                else:
                    wrapped = timed(obj)
                setattr(cls, name, wrapped)


class Job(AutoTimed):
    def do_work(self, n):
        s = 0
        for i in range(n):
            s += i*i
        return s


Job().do_work(10000)


[timed] Job.do_work levou 0.95 ms


333283335000

# Metaclasses vs. `__init_subclass__` vs. ABC



| Aspecto               | Metaclasse                                                                 | \_\_init_subclass\_\_                                                                           | ABCs (*abc.ABC / ABCMeta*)                                                                 |
|-----------------------|----------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------|
| **O que é**           | Uma *classe de uma classe* que controla como as classes são criadas.       | Um método em uma classe base que é executado *após* a criação de uma subclasse.                | Um padrão + ferramentas da stdlib para definir APIs *abstratas*. Implementado com a metaclasse _ABCMeta_. |
| **Quando executa**    | Durante a criação da classe: meta.__new__ → meta.__init__.                 | Logo após o objeto da subclasse existir (pós-criação).                                         | Mesmo tempo que sua metaclasse (_ABCMeta_), mais verificações abstratas no *momento de instanciação*. |
| **Escopo do efeito**  | Global para toda classe que usa essa metaclasse (e suas subclasses).       | Local a uma classe base (e suas subclasses) que define o hook.                                 | Para classes que derivam de _ABC_ (ou usam _ABCMeta_).                                    |
| **Casos de uso típicos** | Impor convenções, registrar classes automaticamente, injetar/transformar atributos/métodos, DSLs. | Validar/aumentar subclasses, registrá-las automaticamente, definir padrões, conexão cooperativa de mixins. | Declarar métodos necessários com @abstractmethod; impedir instanciação até serem implementados. |
| **Poder / invasividade** | Máximo: pode remodelar o dict da classe, bases, anotações, etc.           | Médio: recebe a classe pronta e pode ajustá-la; mais simples e seguro.                         | Focado: principalmente contratos de interface; não destinado a cirurgia pesada de classes. |
| **Facilidade de uso** | Mais difícil de ler/manter; possíveis conflitos de metaclasse.             | Fácil e legível; evita conflitos de metaclasse; coopera via _super()_.                         | Fácil de adotar; padrão familiar para equipes; magia mínima.                              |
| **Herança múltipla**  | Metaclasses diferentes podem colidir → erros de "conflito de metaclasse".  | Funciona bem com MI se cada substituição chamar _super().__init_subclass__.                    | Funciona bem; _ABCMeta_ se mistura com muitas metaclasses se projetado para isso.         |
| **Validação & erros** | Pode levantar erros no *momento de definição da classe*. ↓                 | Pode levantar erros no *momento de definição da subclasse*.                                    | Instanciação bloqueada até que os abstratos sejam implementados (erros no *momento de instanciação*). |
| **Personalização**    | __prepare__, __new__, __init__, reescrita de atributos, injeção de descritores. | Acesso à subclasse finalizada (_cls_), pode inspecionar __dict__, anotações, bases.            | @abstractmethod, @abstractproperty, @abstractclassmethod, @abstractstaticmethod, register() para subclasses virtuais. |
| **Descobribilidade**  | Menos óbvio para leitores não familiarizados com metaprogramação.          | Muito descobrível: um método normal na classe base.                                            | Muito descobrível: base _ABC_ explícita + decoradores abstratos.                          |
| **Desempenho**        | Sobrecarga insignificante, mas a lógica pode ser pesada se fizer muito.    | Sobrecarga insignificante; geralmente mais leve que metaclasses.                               | Insignificante; verificações abstratas são baratas na instanciação.                       |
| **Melhor para**       | Mágica de framework profunda ou transformação uniforme de muitas classes.  | Impor convenções e registro em uma hierarquia sem maquinário pesado.                          | Interfaces públicas/contratos de plug-in com garantias claras de "deve implementar".      |